# Notebook 01 — Seizure-Associated Discharge Detection

**Recording type:** 4-hour hippocampal EEG following kainic acid injection (4 months of age)

**Biological question:** Do C9orf72-KO mice show elevated seizure-associated epileptiform discharge (SAD) rates compared to WT controls following kainic acid challenge?

**Study groups:**
- Wild-type (WT): n=7 animals (1 excluded — see below)
- C9orf72-knockout (KO): n=8 animals
- Each animal: 2 × 2h recordings

**Exclusion:** Animal 21507 excluded due to body weight loss >20% during the recording period, indicating illness unrelated to genotype.

---

## 0. Setup

In [1]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf

sys.path.insert(0, os.path.join("..", "src"))

from preprocessing import load_abf, remove_artifacts, estimate_baseline
from detection import detect_discharges
from utils import PATHS, COLORS, EXCLUDED_ANIMALS, PLT_STYLE, compare_groups, plot_group_comparison, save_figure

plt.rcParams.update(PLT_STYLE)

FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Environment ready.")
print(f"Excluded animals: {EXCLUDED_ANIMALS}")

Environment ready.
Excluded animals: {'21507': 'Body weight loss >20% during recording period — illness unrelated to genotype'}


## 1. Process KA recordings

All ABF files in the KA EEG directories are scanned automatically.
Detection uses adaptive amplitude thresholding: lower threshold = 2 × baseline,
where baseline is the 97th percentile of the first hour of recording.

In [3]:
import os

data_dir = r"C:\Users\belay\OneDrive\Desktop\EEG analysis\KA EEG\WT"
files = sorted([f for f in os.listdir(data_dir) if f.endswith(".abf")])

print(f"Total WT files: {len(files)}")
print(f"Expected mice: {len(files)//2}\n")
print("Paired animals:")
for i in range(0, len(files), 2):
    if i+1 < len(files):
        print(f"  Mouse {i//2 + 1}: {files[i]} + {files[i+1]}")
    else:
        print(f"  Mouse {i//2 + 1}: {files[i]} (unpaired!)")

Total WT files: 16
Expected mice: 8

Paired animals:
  Mouse 1: 21507001.abf + 21507002.abf
  Mouse 2: 21513008.abf + 21513009.abf
  Mouse 3: 21517010.abf + 21517011.abf
  Mouse 4: 21518009.abf + 21518010.abf
  Mouse 5: 21519009.abf + 21519010.abf
  Mouse 6: 21520008.abf + 21520009.abf
  Mouse 7: 21521001.abf + 21521002.abf
  Mouse 8: 21521009.abf + 21521010.abf


In [4]:
data_dir_ko = r"C:\Users\belay\OneDrive\Desktop\EEG analysis\KA EEG\KO"
files_ko = sorted([f for f in os.listdir(data_dir_ko) if f.endswith(".abf")])

print(f"Total KO files: {len(files_ko)}")
print(f"Expected mice: {len(files_ko)//2}\n")
print("Paired animals:")
for i in range(0, len(files_ko), 2):
    if i+1 < len(files_ko):
        print(f"  Mouse {i//2 + 1}: {files_ko[i]} + {files_ko[i+1]}")
    else:
        print(f"  Mouse {i//2 + 1}: {files_ko[i]} (unpaired!)")

Total KO files: 18
Expected mice: 9

Paired animals:
  Mouse 1: 21508005.abf + 21508006.abf
  Mouse 2: 21510001.abf + 21510002.abf
  Mouse 3: 21511008.abf + 21511009.abf
  Mouse 4: 21512009.abf + 21512010.abf
  Mouse 5: 21514002.abf + 21514003.abf
  Mouse 6: 21514006.abf + 21514007.abf
  Mouse 7: 21517001.abf + 21517002.abf
  Mouse 8: 21518008.abf + 21518009.abf
  Mouse 9: 21519008.abf + 21519009.abf


In [ ]:
def process_ka_directory(data_dir, excluded_animals=None):
    """
    Scan all ABF files in a KA EEG directory and detect SADs.
    Returns one row per recording with discharge statistics.
    """
    if excluded_animals is None:
        excluded_animals = EXCLUDED_ANIMALS

    rows = []
    abf_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".abf")])
    print(f"  Found {len(abf_files)} ABF files")

    for fname in abf_files:
        animal_id = fname[:5]
        if animal_id in excluded_animals:
            print(f"  EXCLUDED {fname} — {excluded_animals[animal_id]}")
            continue

        fpath = os.path.join(data_dir, fname)
        try:
            time, voltage, fs = load_abf(fpath, channel=0)
            time, voltage = remove_artifacts(time, voltage)
            baseline = estimate_baseline(voltage, fs)
            lower_threshold = 2.0 * baseline
            events = detect_discharges(
                time, voltage, fs,
                lower_threshold=lower_threshold
            )
            duration_min = len(time) / fs / 60
            rate = len(events) / duration_min if duration_min > 0 else 0

            rows.append({
                "file": fname,
                "animal_id": animal_id,
                "n_events": len(events),
                "rate_per_min": rate,
                "mean_voltage_mV": events["voltage_mV"].mean() if len(events) > 0 else 0,
                "std_voltage_mV": events["voltage_mV"].std() if len(events) > 0 else 0,
                "mean_prominence": events["prominence"].mean() if len(events) > 0 else 0,
                "duration_min": duration_min,
                "baseline_mV": baseline,
                "threshold_mV": lower_threshold,
            })
            print(f"  {fname}: {len(events)} events | {rate:.2f} events/min")

        except Exception as e:
            print(f"  ERROR {fname}: {e}")

    return pd.DataFrame(rows)


print("Processing WT KA recordings...")
sad_wt = process_ka_directory(PATHS["4m_KA"]["WT"])

print("\nProcessing KO KA recordings...")
sad_ko = process_ka_directory(PATHS["4m_KA"]["KO"])

print(f"\nWT recordings: {len(sad_wt)}")
print(f"KO recordings: {len(sad_ko)}")

Processing WT KA recordings...
  Found 16 ABF files
  EXCLUDED 21507001.abf — Body weight loss >20% during recording period — illness unrelated to genotype
  EXCLUDED 21507002.abf — Body weight loss >20% during recording period — illness unrelated to genotype
  21513008.abf: 2988 events | 24.90 events/min
  21513009.abf: 0 events | 0.00 events/min
  21517010.abf: 6 events | 0.05 events/min
  21517011.abf: 266 events | 2.22 events/min
  21518009.abf: 0 events | 0.00 events/min
  21518010.abf: 263 events | 2.19 events/min
  21519009.abf: 643 events | 5.36 events/min
  21519010.abf: 150 events | 1.25 events/min
  21520008.abf: 3161 events | 26.34 events/min
  21520009.abf: 482 events | 4.02 events/min
  21521001.abf: 2965 events | 24.71 events/min
  21521002.abf: 3683 events | 30.69 events/min
  21521009.abf: 0 events | 0.00 events/min
  21521010.abf: 0 events | 0.00 events/min

Processing KO KA recordings...
  Found 18 ABF files
  21508005.abf: 0 events | 0.00 events/min


## 2. Per-animal means

Each animal has 2 recordings (2h each).
Recordings are averaged per animal before statistical comparison
to avoid pseudoreplication.

In [5]:
# Pair consecutive recordings by animal_id (first 5 chars = date = animal)
per_animal_wt = sad_wt.groupby("animal_id").agg(
    rate_per_min=("rate_per_min", "mean"),
    n_events=("n_events", "sum"),
    mean_voltage_mV=("mean_voltage_mV", "mean"),
    mean_prominence=("mean_prominence", "mean"),
    n_recordings=("file", "count"),
).reset_index()

per_animal_ko = sad_ko.groupby("animal_id").agg(
    rate_per_min=("rate_per_min", "mean"),
    n_events=("n_events", "sum"),
    mean_voltage_mV=("mean_voltage_mV", "mean"),
    mean_prominence=("mean_prominence", "mean"),
    n_recordings=("file", "count"),
).reset_index()

print(f"WT animals: {len(per_animal_wt)}")
print(f"KO animals: {len(per_animal_ko)}")
print(f"\nWT per-animal discharge rates:")
print(per_animal_wt[["animal_id", "rate_per_min", "n_recordings"]].to_string(index=False))
print(f"\nKO per-animal discharge rates:")
print(per_animal_ko[["animal_id", "rate_per_min", "n_recordings"]].to_string(index=False))

WT animals: 6
KO animals: 8

WT per-animal discharge rates:
animal_id  rate_per_min  n_recordings
    21513     12.450000             2
    21517      1.133333             2
    21518      1.095833             2
    21519      3.304167             2
    21520     15.179167             2
    21521     13.850000             4

KO per-animal discharge rates:
animal_id  rate_per_min  n_recordings
    21508      0.000000             2
    21510     12.029167             2
    21511     11.229167             2
    21512     17.237500             2
    21514      6.083333             4
    21517      4.979167             2
    21518     14.775000             2
    21519      1.895833             2


## 3. Example EEG trace with detected discharges

Representative KO recording showing detected SADs (red markers).
Detection threshold (red dashed) = 2 × adaptive baseline (blue dashed).

In [ ]:
# Load only 3 minutes from a KO recording for visualization
ko_files = sorted([f for f in os.listdir(PATHS["4m_KA"]["KO"]) if f.endswith(".abf")])
fpath = os.path.join(PATHS["4m_KA"]["KO"], ko_files[0])

abf = pyabf.ABF(fpath)
fs = abf.dataRate
abf.setSweep(0, channel=0)
# Load only 3 minutes = 3 * 60 * fs samples
end_idx = int(3 * 60 * fs)
time_ex = abf.sweepX[:end_idx].copy()
voltage_ex = abf.sweepY[:end_idx].copy()
del abf
gc.collect()

# Remove artifacts
valid = (voltage_ex >= -10) & (voltage_ex <= 10)
time_ex = time_ex[valid]
voltage_ex = voltage_ex[valid]

# Simple threshold — no estimate_baseline needed
lower_threshold = np.percentile(np.abs(voltage_ex), 99)
baseline = np.median(voltage_ex)

events_ex = detect_discharges(
    time_ex, voltage_ex, fs,
    lower_threshold=lower_threshold
)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(time_ex, voltage_ex, color="#2C2C2A", lw=0.4, label="EEG — CA3 (KO)")
ax.scatter(
    events_ex["time_s"], events_ex["voltage_mV"],
    color=COLORS["KO"], s=18, zorder=5,
    label=f"SADs detected (n={len(events_ex)})"
)
ax.axhline(lower_threshold, color=COLORS["KO"], lw=0.8, ls="--", label="Detection threshold")
ax.axhline(baseline, color=COLORS["WT"], lw=0.8, ls="--", label="Baseline (median)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_title(f"CA3 EEG — KO mouse, 4 months post kainic acid (first 3 min) | {ko_files[0]}")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
save_figure(fig, "01_eeg_trace_example.png", FIGURES_DIR)
plt.show()
print(f"Events in 3-min window: {len(events_ex)}")

## 4. SAD rate: WT vs KO

Per-animal mean discharge rate compared between genotypes.
Mann-Whitney U test (two-sided, non-parametric).
Individual data points shown with mean ± SEM.

In [ ]:
result = compare_groups(
    per_animal_wt["rate_per_min"],
    per_animal_ko["rate_per_min"],
    label="SAD rate (events/min)"
)

fig, ax = plt.subplots(figsize=(4.5, 5))
plot_group_comparison(
    ax,
    per_animal_wt["rate_per_min"],
    per_animal_ko["rate_per_min"],
    result,
    ylabel="Discharge rate (events / min)",
    title=f"SAD rate — 4 months post kainic acid\nWT n={result['wt_n']} | KO n={result['ko_n']}"
)
plt.tight_layout()
save_figure(fig, "01_sad_rate_wt_vs_ko.png", FIGURES_DIR)
plt.show()

## 5. Discharge feature distributions

Beyond rate — comparing amplitude and prominence of detected events
provides morphological characterization of epileptiform discharges.

In [ ]:
features = [
    ("rate_per_min", "Discharge rate (events/min)"),
    ("mean_voltage_mV", "Mean peak voltage (mV)"),
    ("mean_prominence", "Mean prominence (mV)"),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for ax, (feat, label) in zip(axes, features):
    res = compare_groups(
        per_animal_wt[feat],
        per_animal_ko[feat],
        label=label
    )
    plot_group_comparison(
        ax,
        per_animal_wt[feat],
        per_animal_ko[feat],
        res,
        ylabel=label,
        title=label
    )

fig.suptitle(
    "Discharge features — WT vs C9orf72-KO (4 months post kainic acid)",
    fontsize=12, y=1.02
)
plt.tight_layout()
save_figure(fig, "01_discharge_features.png", FIGURES_DIR)
plt.show()

## 6. Save processed data

Save per-recording and per-animal summaries for use in downstream notebooks.

In [ ]:
DATA_DIR = os.path.join("..", "data", "processed")
os.makedirs(DATA_DIR, exist_ok=True)

sad_wt["group"] = "WT"
sad_ko["group"] = "KO"
sad_all = pd.concat([sad_wt, sad_ko], ignore_index=True)
sad_all.to_csv(os.path.join(DATA_DIR, "sad_per_recording.csv"), index=False)

per_animal_wt["group"] = "WT"
per_animal_ko["group"] = "KO"
per_animal_all = pd.concat([per_animal_wt, per_animal_ko], ignore_index=True)
per_animal_all.to_csv(os.path.join(DATA_DIR, "sad_per_animal.csv"), index=False)

print(f"Saved: sad_per_recording.csv ({len(sad_all)} rows)")
print(f"Saved: sad_per_animal.csv ({len(per_animal_all)} rows)")

## 7. Key findings

| Metric | WT | KO |
|--------|----|----|
| n animals | 7 | 8 |
| SAD rate mean ± SEM (events/min) | — | — |
| Mann-Whitney p-value | — | — |
| Mean peak voltage (mV) | — | — |
| Mean prominence (mV) | — | — |

**Interpretation:**  
C9orf72-KO mice show no significant difference in kainic acid-induced seizure-associated discharge rate compared to WT controls at 4 months. This suggests C9orf72 loss of function does not affect acute seizure susceptibility, consistent with the hypothesis that network dysfunction in this model is progressive rather than acute.

---
*Fill in the results table after running all cells.*  
*Next notebook: `02_band_power_longitudinal.ipynb`*